In [ ]:
import ee
import os
import pandas as p
import geopandas as gpd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np


In [3]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
ee.Authenticate()



Successfully saved authorization token.


In [4]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [7]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches
from data_utils import make_padded_bbox_all_location, create_shp_for_each_location


In [12]:
import importlib
import data_utils

importlib.reload(data_utils)

from data_utils import create_shp_for_each_location


In [5]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 


site_fp = PROJECT_ROOT / "configs" / "sites.json"
label_fp = PROJECT_ROOT / "configs" / "labels.gpkg"

In [13]:
## Make an .shp for each location from the single labels.gpkg. All the existing code uses the shapefiles or it would be easier not to both

create_shp_for_each_location(label_fp=label_fp, sites_fp= site_fp, project_root= PROJECT_ROOT)

Created Vembanad_points.shp
Created Winam_points.shp
Created Inle_points.shp
Created Hartbeespoort_points.shp
Created Mula_points.shp
Created RawaPening_points.shp
Created Rodman_points.shp
Created Valsequillo_points.shp


/Users/bensutton/Projects/dissertation-code/src/data_utils.py:27: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  location_gdf.to_file(location_fp)
/Users/bensutton/Projects/dissertation-code/.venv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'comparison_dates_used' to 'comparison'
  ogr_write(
/Users/bensutton/Projects/dissertation-code/.venv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 's2_target_image_id' to 's2_target_'
  ogr_write(
/Users/bensutton/Projects/dissertation-code/.venv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 's2_old_image_id' to 's2_old_ima'
  ogr_write(
/Users/bensutton/Projects/dissertation-code/.venv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 's1_image_id' to 's1_image_i'
  ogr_write(
/Users/bensut

In [18]:
# test the created shapefiles
mula_fp = PROJECT_ROOT / "configs/point_files" / "Mula_points.shp"

mula = gpd.read_file(label_fp)

print(mula.columns)

print(mula["class_label"].unique())

Index(['label_id', 'longitude', 'latitude', 'location', 'obs_date',
       'comparison_dates_used', 's2_target_image_id', 's2_old_image_id',
       's1_image_id', 'class_label', 'target_binary', 'notes', 'created_at',
       'updated_at', 'created_by', 'geometry'],
      dtype='object')
['floating_plants' 'open_water' 'LEV' 'surface_algae']


In [59]:
# Make a bounding box for each site based on the total bounds of the labelled points and add a padding

make_padded_bbox_all_location(sites_file= site_fp, project_root= PROJECT_ROOT)

Created padded bbox from the points files for Vembanad,Winam,Inle,Hartbeespoort,Mula,RawaPening,Rodman,Valsequillo


In [7]:
image_collection = create_images_for_all_locations(sites_file= site_fp, project_root= PROJECT_ROOT, clip = True)

Created image collection for: Vembanad
Created image collection for: Winam
Created image collection for: Inle
Created image collection for: Hartbeespoort
Created image collection for: Mula
Created image collection for: RawaPening
Created image collection for: Rodman
Created image collection for: Valsequillo


In [45]:
samples = get_samples(merged_ic=image_collection,sites_file= site_fp, project_root=PROJECT_ROOT)

In [46]:
samples.tail()

,B11,B12,B2,B3,B4,B5,B6,B7,B8,B8A,binary,lat,lc,location,lon,obs_date
1712,2710,2587,1424,1870,2134,2096,2180,2276,2676,2497,1,18.947760,1,Valsequillo,-98.225521,2020-12-27
1713,1610,980,242,448,462,923,1504,1802,1942,2173,1,18.940574,1,Valsequillo,-98.206477,2020-12-27
1714,1653,794,264,623,448,1030,3159,3767,4006,3999,1,18.939406,1,Valsequillo,-98.236660,2020-12-27
1715,1869,1396,513,649,827,949,1144,1317,1359,1513,1,18.934914,1,Valsequillo,-98.196416,2020-12-27
1716,1511,961,224,355,442,661,870,976,1025,1154,1,18.930782,1,Valsequillo,-98.226599,2020-12-27


In [49]:
len(samples.loc[samples["binary"] == 1.0])

461

In [50]:
samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

samples.to_csv(samples_outpath)

In [51]:
samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   B11       116 non-null    int64  
 1   B12       116 non-null    int64  
 2   B2        116 non-null    int64  
 3   B3        116 non-null    int64  
 4   B4        116 non-null    int64  
 5   B5        116 non-null    int64  
 6   B6        116 non-null    int64  
 7   B7        116 non-null    int64  
 8   B8        116 non-null    int64  
 9   B8A       116 non-null    int64  
 10  lat       116 non-null    float64
 11  lc        114 non-null    float64
 12  location  116 non-null    object 
 13  lon       116 non-null    float64
 14  obs_date  116 non-null    object 
dtypes: float64(3), int64(10), object(2)
memory usage: 13.7+ KB


In [ ]:
mula_wh_samples = samples.loc[(samples["location"]=="mula") & (samples["lc"]== 1.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
mula_non_wh_samples = samples.loc[(samples["location"]=="mula")& (samples["lc"]==0.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
bins = np.linspace(0,8000, 20)

for col in ["B11", "B12", "B3", "B4", "B5", "B8"]:
    plt.hist(mula_wh_samples[col], bins, alpha=0.5, label='x')
    plt.hist(mula_non_wh_samples[col], bins, alpha=0.5, label='y')
    plt.legend(loc='upper right')
    plt.title(f"{col}")
    plt.show()



In [15]:
export_patches(merged_ic = image_collection, sites_file= site_fp, project_root=PROJECT_ROOT)

Exporting sampled patches to drive/Dissertation


In [ ]:
import importlib

importlib.reload(gee_utils)
importlib.reload(data_utils)


<module 'gee_utils' from '/Users/bensutton/Library/CloudStorage/Dropbox/MASTERS/Dissertation/code/src/gee_utils.py'>